# 🔥 Análise de Queimadas no Brasil (2015–2024)

**Disciplina:** Linguagem de Programação — Análise e Visualização de Dados com Python  
**Avaliação:** G2 — Projeto de Análise e Visualização de Dados  

---

## 1. Introdução ao Problema

As queimadas representam um dos maiores desafios ambientais do Brasil. Além de destruírem ecossistemas únicos como a Amazônia e o Cerrado, os incêndios causam prejuízos econômicos, afetam a saúde pública e contribuem para as emissões de CO₂.

Este projeto tem como objetivo analisar a evolução temporal e espacial dos focos de queimadas no Brasil entre 2015 e 2024, identificar padrões sazonais, correlações com variáveis climáticas (temperatura, chuva, índice de seca) e classificar o nível de risco por região e bioma.

## 2. Explicação da Base de Dados

A base `simulacao_queimadas_brasil.csv` contém **2.400 registros** com as seguintes variáveis:

| Coluna | Tipo | Descrição |
|---|---|---|
| `ano` | int | Ano do registro |
| `mes` | int | Mês do registro |
| `data` | str | Data no formato YYYY-MM-DD |
| `regiao` | str | Região do Brasil |
| `uf` | str | Unidade Federativa |
| `bioma` | str | Bioma brasileiro |
| `focos_queimada` | int | Número de focos detectados |
| `temperatura_media` | float | Temperatura média (°C) |
| `chuva_mm` | float | Precipitação (mm) |
| `area_atingida_km2` | float | Área atingida (km²) |
| `indice_seca` | float | Índice de seca (0–1) |
| `qualidade_ar` | float | Índice de qualidade do ar |
| `nivel_risco` | str | Classificação de risco |

**Fonte:** Dados simulados com base em metodologia do INPE (Instituto Nacional de Pesquisas Espaciais).

## 3. Leitura dos Dados

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from sqlalchemy import create_engine
import warnings
warnings.filterwarnings('ignore')

# Configurações visuais
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.family'] = 'DejaVu Sans'
sns.set_style('whitegrid')
PALETTE = ['#C0392B', '#E67E22', '#F39C12', '#27AE60', '#2980B9', '#8E44AD']

# Leitura
df = pd.read_csv('../dados/simulacao_queimadas_brasil.csv', parse_dates=['data'])
print(f'Shape: {df.shape}')
df.head()

## 4. Limpeza e Preparação

In [ ]:
# Verificação de valores nulos
print('=== Valores nulos ===')
print(df.isnull().sum())

print('\n=== Duplicatas ===')
print(f'Linhas duplicadas: {df.duplicated().sum()}')

print('\n=== Tipos de dados ===')
print(df.dtypes)

In [ ]:
# Verificação de outliers com IQR
for col in ['focos_queimada', 'area_atingida_km2', 'indice_seca']:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    outliers = df[(df[col] < Q1 - 1.5*IQR) | (df[col] > Q3 + 1.5*IQR)]
    print(f'{col}: {len(outliers)} outliers detectados')

## 5. Engenharia de Atributos

In [ ]:
# Novas colunas derivadas
df['mes_nome'] = df['data'].dt.month_name()
df['trimestre'] = df['data'].dt.quarter.map({1: 'T1', 2: 'T2', 3: 'T3', 4: 'T4'})

# Classificação binária de seca severa
df['seca_severa'] = (df['indice_seca'] >= 0.7).astype(int)

# Razão entre área atingida e focos
df['area_por_foco'] = df['area_atingida_km2'] / df['focos_queimada'].replace(0, np.nan)

# Nível de risco como ordinal
risco_map = {'Baixo': 1, 'Médio': 2, 'Alto': 3, 'Crítico': 4}
df['risco_ord'] = df['nivel_risco'].map(risco_map)

print('Atributos criados com sucesso!')
df[['mes_nome', 'trimestre', 'seca_severa', 'area_por_foco', 'risco_ord']].head()

## 6. Análise Exploratória

In [ ]:
print('=== Estatísticas Descritivas ===')
df[['focos_queimada', 'temperatura_media', 'chuva_mm',
    'area_atingida_km2', 'indice_seca', 'qualidade_ar']].describe().round(2)

In [ ]:
# Distribuição de focos por região
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

focos_regiao = df.groupby('regiao')['focos_queimada'].sum().sort_values(ascending=False)
focos_regiao.plot(kind='bar', ax=axes[0], color=PALETTE, edgecolor='white', linewidth=0.5)
axes[0].set_title('Total de Focos por Região', fontweight='bold')
axes[0].set_xlabel('')
axes[0].tick_params(axis='x', rotation=30)

focos_bioma = df.groupby('bioma')['focos_queimada'].sum().sort_values()
focos_bioma.plot(kind='barh', ax=axes[1], color=PALETTE[::-1], edgecolor='white')
axes[1].set_title('Total de Focos por Bioma', fontweight='bold')
axes[1].set_xlabel('Focos de Queimada')

plt.suptitle('Distribuição Espacial das Queimadas', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../imagens/focos_regiao_bioma.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Evolução temporal
focos_ano = df.groupby('ano')['focos_queimada'].sum()

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(focos_ano.index, focos_ano.values, marker='o', color='#C0392B',
        linewidth=2.5, markersize=8, markerfacecolor='white', markeredgewidth=2)
ax.fill_between(focos_ano.index, focos_ano.values, alpha=0.15, color='#C0392B')
ax.set_title('Evolução Anual de Focos de Queimada (2015–2024)', fontsize=13, fontweight='bold')
ax.set_xlabel('Ano')
ax.set_ylabel('Total de Focos')
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.savefig('../imagens/evolucao_anual.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Sazonalidade mensal
focos_mes = df.groupby('mes')['focos_queimada'].mean()
meses_abrev = ['Jan','Fev','Mar','Abr','Mai','Jun','Jul','Ago','Set','Out','Nov','Dez']

fig, ax = plt.subplots(figsize=(12, 4))
bars = ax.bar(meses_abrev, focos_mes.values,
              color=['#C0392B' if v == focos_mes.max() else '#E8A598' for v in focos_mes.values],
              edgecolor='white', linewidth=0.8)
ax.set_title('Sazonalidade Mensal — Média de Focos de Queimada', fontsize=13, fontweight='bold')
ax.set_xlabel('Mês')
ax.set_ylabel('Média de Focos')
plt.tight_layout()
plt.savefig('../imagens/sazonalidade_mensal.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. KPIs

In [ ]:
kpis = {
    'Total de Focos'           : df['focos_queimada'].sum(),
    'Área Total Atingida (km²)': df['area_atingida_km2'].sum().round(2),
    'Temperatura Média (°C)'   : df['temperatura_media'].mean().round(2),
    'Precipitação Média (mm)'  : df['chuva_mm'].mean().round(2),
    'Índice de Seca Médio'     : df['indice_seca'].mean().round(3),
    'Eventos Críticos (%)'     : round((df['nivel_risco'] == 'Crítico').mean() * 100, 2),
    'Ano com Mais Focos'       : int(df.groupby('ano')['focos_queimada'].sum().idxmax()),
    'Bioma Mais Afetado'       : df.groupby('bioma')['focos_queimada'].sum().idxmax(),
    'Região Mais Afetada'      : df.groupby('regiao')['focos_queimada'].sum().idxmax(),
    'UF com Mais Focos'        : df.groupby('uf')['focos_queimada'].sum().idxmax(),
}

pd.DataFrame(kpis.items(), columns=['KPI', 'Valor'])

## 8. Gráficos — Correlação e Risco

In [ ]:
# Mapa de correlação
numeric_cols = ['focos_queimada', 'temperatura_media', 'chuva_mm',
                'area_atingida_km2', 'indice_seca', 'qualidade_ar']
corr = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(8, 6))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn_r',
            linewidths=0.5, ax=ax, mask=mask, annot_kws={'size': 10},
            square=True, cbar_kws={'shrink': 0.8})
ax.set_title('Mapa de Correlação — Variáveis Climáticas e Queimadas',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('../imagens/correlacao.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Nível de risco por bioma
risco_ordem = ['Baixo', 'Médio', 'Alto', 'Crítico']
risco_bioma = df.groupby(['bioma', 'nivel_risco']).size().unstack(fill_value=0)[risco_ordem]

cores_risco = ['#2ECC71', '#F39C12', '#E67E22', '#C0392B']
risco_bioma.plot(kind='bar', stacked=True, color=cores_risco,
                 figsize=(12, 5), edgecolor='white', linewidth=0.5)
plt.title('Distribuição de Nível de Risco por Bioma', fontsize=13, fontweight='bold')
plt.xlabel('')
plt.ylabel('Número de Registros')
plt.xticks(rotation=30, ha='right')
plt.legend(title='Nível de Risco', bbox_to_anchor=(1.01, 1))
plt.tight_layout()
plt.savefig('../imagens/risco_bioma.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Scatter: seca vs focos
fig, ax = plt.subplots(figsize=(10, 5))
for i, regiao in enumerate(df['regiao'].unique()):
    sub = df[df['regiao'] == regiao]
    ax.scatter(sub['indice_seca'], sub['focos_queimada'],
               label=regiao, alpha=0.5, s=30, color=PALETTE[i])

# Linha de tendência geral
z = np.polyfit(df['indice_seca'], df['focos_queimada'], 1)
p = np.poly1d(z)
xs = np.linspace(df['indice_seca'].min(), df['indice_seca'].max(), 100)
ax.plot(xs, p(xs), 'k--', linewidth=1.5, label='Tendência geral')

ax.set_title('Índice de Seca vs. Focos de Queimada por Região', fontsize=13, fontweight='bold')
ax.set_xlabel('Índice de Seca')
ax.set_ylabel('Focos de Queimada')
ax.legend(bbox_to_anchor=(1.01, 1))
plt.tight_layout()
plt.savefig('../imagens/seca_vs_focos.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Persistência em Banco de Dados (SQLite)

In [ ]:
import os
os.makedirs('../database', exist_ok=True)

engine = create_engine('sqlite:///../database/queimadas.db')
df.to_sql('queimadas', engine, if_exists='replace', index=False)
print('Dados persistidos no banco SQLite com sucesso!')

# Consulta SQL de verificação
consulta = pd.read_sql("""
    SELECT regiao,
           SUM(focos_queimada)     AS total_focos,
           ROUND(AVG(indice_seca), 3) AS media_seca,
           COUNT(*)                AS registros
    FROM queimadas
    GROUP BY regiao
    ORDER BY total_focos DESC
""", engine)
consulta

## 10. Interpretação dos Resultados e Conclusão

### Principais Achados

1. **Sazonalidade acentuada:** Os meses de agosto a outubro concentram os maiores índices de focos de queimada, coincidindo com o período de seca no Centro-Oeste e Norte do Brasil.

2. **Concentração regional:** A região **Norte** lidera em número absoluto de focos, com destaque para Pará (PA) e Amazonas (AM), enquanto o **Cerrado** e a **Amazônia** são os biomas mais atingidos em área.

3. **Correlação climática confirmada:** A análise de correlação evidencia relação positiva entre `indice_seca` e `focos_queimada` (quanto maior a seca, mais focos), e relação negativa entre `chuva_mm` e `area_atingida_km2` (mais chuva, menor área destruída).

4. **Qualidade do ar:** Eventos de maior intensidade de queimada (nível Crítico) coincidem com piora no índice de qualidade do ar, afetando populações locais.

5. **Tendência temporal:** O período apresenta variações anuais expressivas, com picos nos anos de El Niño, que intensificam as secas na região amazônica.

### Conclusão

O monitoramento de variáveis climáticas — especialmente o índice de seca e a temperatura — demonstrou ser um forte preditor do risco de queimadas. A adoção de sistemas de alerta precoce baseados nessas métricas, aliada ao fortalecimento da fiscalização em biomas prioritários como a Amazônia e o Cerrado durante o segundo semestre, representa a estratégia mais eficaz para a redução dos danos ambientais e sociais causados pelas queimadas no Brasil.